In [0]:
!pip install catboost

In [0]:
%restart_python

In [0]:
import numpy as np
import pandas as pd

import mlflow
import mlflow.catboost

from catboost import CatBoostClassifier

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    brier_score_loss,
)

from mlflow.models import infer_signature

from pyspark.sql import functions as F

In [0]:
TRAINING_TABLE = (
    "high_garden.gold.forecasting_features"
)

SILVER_TABLE = (
    "high_garden.silver.coffee_consumption"
)

OUTPUT_TABLE = (
    "high_garden.gold.growth_predictions"
)

METRICS_TABLE = (
    "high_garden.gold.growth_classifier_metrics"
)

EXPERIMENT_NAME = (
    "/Shared/high-garden-coffee"
)

In [0]:
sdf = spark.table(
    TRAINING_TABLE
)

print(
    "Rows:",
    sdf.count()
)

display(
    sdf.limit(10)
)

In [0]:
pdf = sdf.toPandas()

print(
    pdf.shape
)

print(
    pdf.columns.tolist()
)

In [0]:
pdf[
    "growth_target"
] = (
    pdf["target"]
    >
    pdf["lag_1"]
).astype(int)

In [0]:
class_distribution = (
    pdf[
        "growth_target"
    ]
    .value_counts(
        normalize=True
    )
    .sort_index()
)

print(
    class_distribution
)

In [0]:
print(
    pdf[
        "growth_target"
    ].value_counts()
)

In [0]:
categorical_features = [
    "country",
    "coffee_type",
]

numeric_features = [
    "start_year",

    "lag_1",
    "lag_2",
    "lag_3",
    "lag_5",

    "rolling_mean_3",
    "rolling_mean_5",
    "rolling_std_3",

    "historical_growth_1y",
    "lag_1_zero",
]

feature_columns = (
    categorical_features
    +
    numeric_features
)

In [0]:
for column in categorical_features:

    pdf[column] = (
        pdf[column]
        .astype(str)
    )

In [0]:
required_history = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_5",
    "rolling_mean_3",
    "rolling_mean_5",
]

model_df = (
    pdf
    .dropna(
        subset=
        required_history
        +
        ["growth_target"]
    )
    .copy()
)

In [0]:
print(
    "Original rows:",
    len(pdf)
)

print(
    "Model-ready rows:",
    len(model_df)
)

In [0]:
assert (
    model_df[
        "growth_target"
    ]
    .isna()
    .sum()
    == 0
)

assert set(
    model_df[
        "growth_target"
    ].unique()
).issubset(
    {0, 1}
)

print(
    "Classification dataset validation passed."
)

In [0]:
BACKTEST_YEARS = [
    2015,
    2016,
    2017,
    2018,
    2019,
]

In [0]:
CLASSIFIER_PARAMS = {
    "iterations": 400,
    "depth": 5,
    "learning_rate": 0.03,
    "loss_function": "Logloss",
    "random_seed": 42,
    "verbose": False,
}

In [0]:
fold_metrics = []
backtest_predictions = []

for test_year in BACKTEST_YEARS:

    train_df = model_df[
        model_df["start_year"]
        < test_year
    ].copy()

    test_df = model_df[
        model_df["start_year"]
        == test_year
    ].copy()

    assert len(train_df) > 0
    assert len(test_df) > 0

    X_train = train_df[
        feature_columns
    ]

    y_train = train_df[
        "growth_target"
    ]

    X_test = test_df[
        feature_columns
    ]

    y_test = test_df[
        "growth_target"
    ]

    classifier = (
        CatBoostClassifier(
            **CLASSIFIER_PARAMS
        )
    )

    classifier.fit(
        X_train,
        y_train,
        cat_features=
            categorical_features,
    )

    growth_probability = (
        classifier
        .predict_proba(
            X_test
        )[:, 1]
    )

    predicted_growth = (
        growth_probability
        >= 0.5
    ).astype(int)

    # ROC-AUC requires both classes
    if y_test.nunique() > 1:

        roc_auc = (
            roc_auc_score(
                y_test,
                growth_probability
            )
        )

    else:

        roc_auc = np.nan

    f1 = f1_score(
        y_test,
        predicted_growth,
        zero_division=0
    )

    precision = precision_score(
        y_test,
        predicted_growth,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predicted_growth,
        zero_division=0
    )

    brier = brier_score_loss(
        y_test,
        growth_probability
    )

    fold_metrics.append(
        {
            "year":
                test_year,

            "roc_auc":
                roc_auc,

            "f1":
                f1,

            "precision":
                precision,

            "recall":
                recall,

            "brier":
                brier,
        }
    )

    fold_output = (
        test_df[
            [
                "country",
                "coffee_type",
                "start_year",
                "growth_target",
            ]
        ]
        .copy()
    )

    fold_output[
        "growth_probability"
    ] = growth_probability

    fold_output[
        "predicted_growth"
    ] = predicted_growth

    backtest_predictions.append(
        fold_output
    )

In [0]:
classifier_metrics = (
    pd.DataFrame(
        fold_metrics
    )
)

display(
    classifier_metrics
)

In [0]:
classifier_summary = {
    "roc_auc":
        classifier_metrics[
            "roc_auc"
        ].mean(
            skipna=True
        ),

    "f1":
        classifier_metrics[
            "f1"
        ].mean(),

    "precision":
        classifier_metrics[
            "precision"
        ].mean(),

    "recall":
        classifier_metrics[
            "recall"
        ].mean(),

    "brier":
        classifier_metrics[
            "brier"
        ].mean(),
}

print(
    classifier_summary
)

In [0]:
metrics_output = pd.DataFrame(
    [
        {
            "model":
                "catboost_growth_classifier",

            **classifier_summary,
        }
    ]
)

metrics_sdf = (
    spark.createDataFrame(
        metrics_output
    )
)

(
    metrics_sdf.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        METRICS_TABLE
    )
)

In [0]:
X_full = model_df[
    feature_columns
]

y_full = model_df[
    "growth_target"
]

final_classifier = (
    CatBoostClassifier(
        **CLASSIFIER_PARAMS
    )
)

final_classifier.fit(
    X_full,
    y_full,
    cat_features=
        categorical_features,
)

In [0]:
mlflow.set_experiment(
    EXPERIMENT_NAME
)

In [0]:
input_example = (
    X_full.head(5)
)

output_example = (
    final_classifier
    .predict_proba(
        input_example
    )[:, 1]
)

signature = infer_signature(
    input_example,
    output_example
)

In [0]:
with mlflow.start_run(
    run_name="catboost_growth_classifier"
) as run:

    for key, value in (
        CLASSIFIER_PARAMS.items()
    ):

        mlflow.log_param(
            key,
            value
        )

    mlflow.log_param(
        "validation",
        "rolling_origin"
    )

    mlflow.log_param(
        "threshold",
        0.5
    )

    for metric, value in (
        classifier_summary.items()
    ):

        if not np.isnan(value):

            mlflow.log_metric(
                metric,
                float(value)
            )

    mlflow.catboost.log_model(
        final_classifier,
        name="model",
        input_example=
            input_example,
        signature=
            signature,
    )

    mlflow.set_tag(
        "task",
        "growth_classification"
    )

    classifier_run_id = (
        run.info.run_id
    )

print(
    "Growth classifier run:",
    classifier_run_id
)

In [0]:
silver_df = spark.table(
    SILVER_TABLE
)

latest_year = (
    silver_df
    .agg(
        F.max(
            "start_year"
        ).alias(
            "latest_year"
        )
    )
    .first()[
        "latest_year"
    ]
)

forecast_year = (
    latest_year + 1
)

print(
    "Latest observed:",
    latest_year
)

print(
    "Growth prediction year:",
    forecast_year
)

In [0]:
history_pdf = (
    silver_df
    .select(
        "country",
        "coffee_type",
        "start_year",
        "domestic_consumption"
    )
    .toPandas()
)

In [0]:
future_rows = []

for (
    country,
    coffee_type
), group in history_pdf.groupby(
    [
        "country",
        "coffee_type"
    ]
):

    group = (
        group
        .sort_values(
            "start_year"
        )
    )

    values = (
        group[
            "domestic_consumption"
        ]
        .astype(float)
        .to_numpy()
    )

    if len(values) < 5:
        continue

    lag_1 = values[-1]
    lag_2 = values[-2]
    lag_3 = values[-3]
    lag_5 = values[-5]

    rolling_mean_3 = (
        values[-3:].mean()
    )

    rolling_mean_5 = (
        values[-5:].mean()
    )

    rolling_std_3 = (
        values[-3:].std(
            ddof=1
        )
    )

    if lag_2 > 0:

        historical_growth_1y = (
            lag_1 - lag_2
        ) / lag_2

    else:

        historical_growth_1y = np.nan

    lag_1_zero = int(
        lag_1 == 0
    )

    future_rows.append(
        {
            "country":
                str(country),

            "coffee_type":
                str(coffee_type),

            "start_year":
                forecast_year,

            "lag_1":
                lag_1,

            "lag_2":
                lag_2,

            "lag_3":
                lag_3,

            "lag_5":
                lag_5,

            "rolling_mean_3":
                rolling_mean_3,

            "rolling_mean_5":
                rolling_mean_5,

            "rolling_std_3":
                rolling_std_3,

            "historical_growth_1y":
                historical_growth_1y,

            "lag_1_zero":
                lag_1_zero,
        }
    )

In [0]:
future_df = pd.DataFrame(
    future_rows
)

print(
    future_df.shape
)

display(
    future_df.head(10)
)

In [0]:
assert (
    future_df[
        [
            "lag_1",
            "lag_2",
            "lag_3",
            "lag_5",
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

print(
    "Future feature validation passed."
)

In [0]:
future_X = (
    future_df[
        feature_columns
    ]
)

future_df[
    "growth_probability"
] = (
    final_classifier
    .predict_proba(
        future_X
    )[:, 1]
)

In [0]:
future_df[
    "predicted_growth"
] = (
    future_df[
        "growth_probability"
    ]
    >= 0.5
).astype(int)

In [0]:
future_df[
    "crop_year"
] = (
    str(forecast_year)
    +
    "/"
    +
    str(
        forecast_year + 1
    )[-2:]
)

In [0]:
growth_output = future_df[
    [
        "country",
        "coffee_type",
        "crop_year",
        "start_year",
        "growth_probability",
        "predicted_growth",
    ]
].copy()

In [0]:
growth_output = (
    growth_output
    .sort_values(
        "growth_probability",
        ascending=False
    )
)

In [0]:
display(
    growth_output
)

In [0]:
assert (
    len(growth_output)
    == 55
)

assert (
    growth_output[
        "growth_probability"
    ]
    .between(
        0,
        1
    )
    .all()
)

print(
    "Growth inference validation passed."
)

In [0]:
growth_sdf = (
    spark.createDataFrame(
        growth_output
    )
)

growth_sdf = (
    growth_sdf
    .withColumn(
        "generated_at",
        F.current_timestamp()
    )
)

(
    growth_sdf.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        OUTPUT_TABLE
    )
)

In [0]:
%sql

SELECT
    country,
    coffee_type,
    crop_year,
    ROUND(
        growth_probability * 100,
        2
    ) AS growth_probability_pct,
    predicted_growth
FROM high_garden.gold.growth_predictions
ORDER BY growth_probability DESC;